# AAT6153 Chapter 2 — Demo
## Recurrent Networks: SimpleRNN, LSTM and GRU from the Inside
**Slides:** Recurrent Neural Networks (49–57), LSTM (58–61), GRU (62), Comparing Recurrent Layers (63), Stacked and Bidirectional RNNs (64–65)

In this notebook you will:
1. Write the **RNN forward pass** as a NumPy loop and check it against Keras `SimpleRNN` (slides 50–53).
2. See **vanishing and exploding gradients** as a product of factors (slide 56).
3. Implement one **LSTM** using the equations on slide 60 and check it against Keras `LSTM`.
4. Compare **parameter counts** of SimpleRNN, LSTM and GRU (slide 63).
5. Build **stacked** and **bidirectional** models (slides 64–65) and set up **gradient clipping** (slide 57).

*Runtime: CPU is fine for this notebook.*

---
## RNN forward pass in NumPy (slide 52)
h<sub>t</sub> = tanh(W<sub>x</sub> x<sub>t</sub> + W<sub>h</sub> h<sub>t−1</sub> + b): the **same weights** at every time step.

In [ ]:
import numpy as np

timesteps, input_dim, units = 10, 8, 16
inputs = np.random.random((timesteps, input_dim))   # one input sequence

W_x = np.random.random((units, input_dim))    # input  -> hidden weights
W_h = np.random.random((units, units))        # hidden -> hidden weights
b   = np.random.random((units,))
h_t = np.zeros((units,))                      # initial hidden state

outputs = []
for x_t in inputs:                            # loop over the time steps
    h_t = np.tanh(W_x @ x_t + W_h @ h_t + b)  # same weights at every step
    outputs.append(h_t)
outputs = np.stack(outputs)                   # (timesteps, units)
print(outputs.shape)

### The same computation in Keras
We copy our weights into a Keras `SimpleRNN` layer. Keras stores them transposed: `kernel` is (input_dim, units) and `recurrent_kernel` is (units, units).

In [ ]:
import keras
from keras import layers

rnn = layers.SimpleRNN(units, return_sequences=True)
rnn.build((None, timesteps, input_dim))
rnn.set_weights([W_x.T, W_h.T, b])

keras_out = rnn(inputs[None].astype("float32")).numpy()[0]
print("Max difference NumPy vs Keras:", np.abs(keras_out - outputs).max())

## Recurrent layers in Keras: `return_sequences` (slide 53)

In [ ]:
x = np.random.random((32, 10, 8)).astype("float32")  # (batch, timesteps, features)

print(layers.SimpleRNN(16)(x).shape)                         # (32, 16)
print(layers.SimpleRNN(16, return_sequences=True)(x).shape)  # (32, 10, 16)

---
## Vanishing and exploding gradients (slide 56)
The gradient that flows from step *t* back to step *t − k* is multiplied by roughly the same factor *k* times. Below we multiply the RNN's step-to-step Jacobians and watch the gradient size.

In [ ]:
import matplotlib.pyplot as plt

def gradient_norms(scale, steps=50, units=16, seed=0):
    rng = np.random.default_rng(seed)
    W_h = rng.normal(size=(units, units))
    W_h *= scale / np.max(np.abs(np.linalg.eigvals(W_h)))   # set the spectral radius
    h, J = np.zeros(units), np.eye(units)
    norms = []
    for _ in range(steps):
        h = np.tanh(W_h @ h + rng.normal(scale=0.1, size=units))
        J = (np.diag(1 - h**2) @ W_h) @ J        # chain rule: d h_t / d h_0
        norms.append(np.linalg.norm(J))
    return norms

for scale, label in [(0.5, "factor < 1: vanishing"), (1.0, "factor ≈ 1"), (2.5, "factor > 1: exploding")]:
    plt.semilogy(gradient_norms(scale), label=label)
plt.xlabel("Steps back in time (k)"); plt.ylabel("‖∂h_t / ∂h_(t−k)‖  (log scale)")
plt.title("Gradient size through a SimpleRNN"); plt.legend(); plt.show()

With a factor below 1 the gradient shrinks exponentially, so early words barely influence learning. With tanh, very large weights also saturate the units, so explosions are bounded here, but they still make training unstable in practice.

### Remedy for exploding gradients: gradient clipping (slide 57)

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=1e-3,
                                  clipnorm=1.0)   # rescale if ||g|| > 1
print(optimizer.get_config()["clipnorm"])

---
## The LSTM cell, equation by equation (slides 59–60)
| Gate | Equation |
|---|---|
| Forget | f<sub>t</sub> = σ(W<sub>f</sub>[h<sub>t−1</sub>, x<sub>t</sub>] + b<sub>f</sub>) |
| Input | i<sub>t</sub> = σ(W<sub>i</sub>[h<sub>t−1</sub>, x<sub>t</sub>] + b<sub>i</sub>) |
| Candidate | c̃<sub>t</sub> = tanh(W<sub>c</sub>[h<sub>t−1</sub>, x<sub>t</sub>] + b<sub>c</sub>) |
| Cell update | c<sub>t</sub> = f<sub>t</sub> ⊙ c<sub>t−1</sub> + i<sub>t</sub> ⊙ c̃<sub>t</sub> |
| Output | o<sub>t</sub> = σ(W<sub>o</sub>[h<sub>t−1</sub>, x<sub>t</sub>] + b<sub>o</sub>) |
| Hidden state | h<sub>t</sub> = o<sub>t</sub> ⊙ tanh(c<sub>t</sub>) |

We take the (random) weights of a Keras `LSTM` layer and run the equations ourselves. Keras packs the four blocks side by side in the order **input, forget, candidate, output**.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

units = 4
lstm = layers.LSTM(units, return_sequences=True)
seq = np.random.random((1, 6, 3)).astype("float32")     # 1 sequence, 6 steps, 3 features
keras_h = lstm(seq).numpy()[0]

kernel, recurrent_kernel, bias = lstm.get_weights()     # (3, 16), (4, 16), (16,)
h, c = np.zeros(units), np.zeros(units)
for x_t in seq[0]:
    z = x_t @ kernel + h @ recurrent_kernel + bias
    i_t, f_t, c_hat, o_t = np.split(z, 4)
    i_t, f_t, o_t = sigmoid(i_t), sigmoid(f_t), sigmoid(o_t)
    c = f_t * c + i_t * np.tanh(c_hat)                  # cell update (the conveyor belt)
    h = o_t * np.tanh(c)                                # hidden state
    print(f"forget gate {f_t.round(2)}   input gate {i_t.round(2)}   h {h.round(3)}")

print("\nMax difference NumPy vs Keras (last step):", np.abs(h - keras_h[-1]).max())

---
## Comparing SimpleRNN, LSTM and GRU (slide 63)

In [ ]:
for rnn in [layers.SimpleRNN(32), layers.LSTM(32), layers.GRU(32)]:
    model = keras.Sequential([keras.Input(shape=(None, 8)), rnn])
    print(f"{rnn.__class__.__name__:10s} {model.count_params():>6,}")

u, m = 32, 8
print(f"\nFormulas: SimpleRNN u(u+m+1) = {u*(u+m+1):,}   LSTM 4u(u+m+1) = {4*u*(u+m+1):,}   GRU 3u(u+m+2) = {3*u*(u+m+2):,}")

## Stacked RNNs (slide 64)
Every recurrent layer except the last needs `return_sequences=True`.

In [ ]:
vocab_size = 10000
model = keras.Sequential([
    keras.Input(shape=(None,), dtype="int32"),
    layers.Embedding(vocab_size, 64, mask_zero=True),
    layers.LSTM(64, return_sequences=True),   # full sequence to next layer
    layers.LSTM(32),                          # last hidden state only
    layers.Dense(1, activation="sigmoid")])
model.summary()

## Bidirectional RNNs (slide 65)

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(None,), dtype="int32"),
    layers.Embedding(vocab_size, 64, mask_zero=True),
    layers.Bidirectional(layers.LSTM(32)),    # forward + backward = 64
    layers.Dense(1, activation="sigmoid")])
print(model.output_shape, model.layers[1].output.shape)

---
## Try it yourself
1. Change `units` in the SimpleRNN check to 64. Does NumPy still match Keras?
2. Write the **GRU** update h<sub>t</sub> = z<sub>t</sub> ⊙ h<sub>t−1</sub> + (1 − z<sub>t</sub>) ⊙ h̃<sub>t</sub> (slide 62) in words: what happens when the update gate is 1?
3. Add `dropout=0.2, recurrent_dropout=0.2` to an LSTM (slide 66). Does the parameter count change?

In [ ]:
# Use this cell for your experiments